# Milestone 2: direct shape-grounding sanity experiment

Train fresh weights on **What color is the circle/square/triangle?** using the original three-object images. Original rendering, layouts, vocabulary, model size and R=2 stay fixed. The only task change is removing neighbor selection. No baseline weights or test examples are used.

Enable a GPU and internet, attach the original baseline archive, and push these changes to GitHub before running. This uses one GPU (`cuda:0`) and preserves Kaggle's PyTorch installation. Training runs for exactly **2,880 updates / 92,160 QA presentations**, followed by frozen train/validation diagnostics and validation image/question controls. Shorter sequences mean this is not an exact compute match.

See `docs/milestones/milestone2_shape_grounding.md` for the preregistered protocol and interpretation limits. A fresh output directory is required; training resume is not supported in this increment.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # A committed revision containing this notebook; commit SHA preferred.
REPO_DIR = "/kaggle/working/multi-modal-loop-shape-grounding"
RUN_ROOT = "/kaggle/working/milestone2_shape_grounding"
BASELINE_SOURCE = "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_baseline_artifacts.zip"

## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit the source and record the protocol

The original archive is audited before training. Its manifest supplies the exact images and splits; model weights start fresh.


In [ ]:
import multimodal_loop.eval.kaggle_shape_grounding as helpers
from multimodal_loop.eval.kaggle_shape_grounding import (
    archive_shape_grounding,
    prepare_shape_grounding,
    run_shape_grounding,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_shape_grounding(REPO_DIR, RUN_ROOT, BASELINE_SOURCE)

## Train and evaluate

13 complete passes plus 72 batches of pass 14; no early stopping or checkpoint selection. Frozen evaluation covers 6,912 training and 1,728 validation questions. Controls keep recipient targets fixed. Logs stream below and are retained.


In [ ]:
report = run_shape_grounding(run)

## Inspect the results

Training criteria: at least 95% accuracy for each shape. Held-out direct grounding: at least 90% for each shape and 30 percentage-point overall gaps against blank images and both mean shuffle controls. These are diagnostic criteria, not Milestone 2 completion gates.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

for split, metrics in report["splits"].items():
    print(split, json.dumps(metrics, indent=2))
print(json.dumps(report["assessment"], indent=2))
controls = json.loads((run.root / "diagnosis" / "controls.json").read_text())
print("Validation control gaps:", controls["results"]["accuracy_gaps"])
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))

## Retain artifacts

Download the archive and bring it back for review. It includes the checkpoint, manifest, protocol, metrics, all train/validation predictions, rendered error preview, controls, hashes, runtime/revision provenance and logs. Staged baseline artifacts are excluded.


In [ ]:
archive = archive_shape_grounding(run)
print("Shape-grounding archive:", archive)
display(FileLink(str(archive)))